# Realism tools via `lic_dsf.realism`

Realism 4 fiscal adjustment, Realism 2 multiplier, and Output 4 panels.
See `docs/09-realism.md`.


In [ ]:
from __future__ import annotations

from pathlib import Path

from lic_dsf.dsa import BaselinePublicBook
from lic_dsf.pv import (
    ExternalDebtBook,
    MacroDebtBook,
    PVPortfolio,
    load_external_debt_inputs,
    load_instruments_from_workbook,
    load_lc_nr_instruments_from_workbook,
    load_macro_debt_inputs,
)
from lic_dsf.realism import (
    fiscal_adjustment_panel,
    fiscal_multiplier_panel,
    place_in_lic_histogram,
    placement_summary,
    projected_three_year_adjustment,
)

REPO = Path("..").resolve() if Path("data").exists() is False else Path(".")
# Prefer repo root when run from demo/
for cand in (Path.cwd(), Path.cwd().parent):
    if (cand / "data" / "lic-dsf-template-2025-08-12.xlsx").exists():
        REPO = cand
        break
WB = REPO / "data" / "lic-dsf-template-2025-08-12.xlsx"

instruments = load_instruments_from_workbook(WB, include_zero_disbursement=True)
lc_nr = load_lc_nr_instruments_from_workbook(WB, include_zero_disbursement=True)
ext = ExternalDebtBook(
    portfolio=PVPortfolio(instruments=tuple(instruments) + tuple(lc_nr)),
    inputs=load_external_debt_inputs(WB),
)
macro = MacroDebtBook(inputs=load_macro_debt_inputs(WB), external=ext)
pub = BaselinePublicBook(macro=macro, external=ext)

pd_pct = pub.primary_deficit_to_gdp()
first = macro.inputs.first_projection_year
projected = projected_three_year_adjustment(pd_pct, first)
print("Projected 3-yr adjustment:", projected)
print(placement_summary(place_in_lic_histogram(projected)))
fiscal_adjustment_panel(pd_pct, first).head()



In [ ]:
pb_pct = 100.0 * macro.primary_balance() / macro.gdp_lcu()
fiscal_multiplier_panel(pb_pct, macro.real_gdp_growth(), first).loc[:, ("impact", 0.2)].head()

